 ### Finviz Stock Data Scraper  

 Compatible with both local PC (VS Code) and Google Colab environments.  

Scrape 1360 (20 x 68) = tickers (stocks and etfs), sorted by:  
- Market Cap. Desc  
- price > $5  
- Average Volume (avg number of shares traded per day) > 2M  

In [ ]:
import datetime
from io import StringIO
from pathlib import Path
import random
import sys
import time
import zipfile

from curl_cffi import requests
import pandas as pd
import pytz

# Detect execution environment
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    output_dir = Path(".")
    print("Running in Google Colab environment.")
else:
    output_dir = Path.home() / "Downloads"
    print(f"Running on local PC. Output directory: {output_dir.resolve()}")

output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Generate single set of screener URLs incrementing r by 20 from 1 to 1341
base_url = (
    "https://finviz.com/screener?v=111&f=sh_avgvol_o2000,sh_price_o5&o=-marketcap&r="
)
urls = [f"{base_url}{r}" for r in range(1, 1342, 20)]

print(f"Generated {len(urls)} total URLs to scrape.")

In [ ]:
def scrape_finviz_curl(url: str) -> pd.DataFrame | None:
    try:
        response = requests.get(url, impersonate="chrome120")
        if response.status_code != 200:
            print(f"Failed HTTP Request. Status Code: {response.status_code}")
            return None

        dfs = pd.read_html(StringIO(response.text))
        screener_df = max(dfs, key=len)

        if "Ticker" not in screener_df.columns:
            screener_df.columns = screener_df.iloc[0]
            screener_df = screener_df[1:]

        return screener_df.reset_index(drop=True)

    except Exception as e:
        print(f"Error scraping URL ({url}): {e}")
        return None

In [ ]:
total_urls = len(urls)
df = pd.DataFrame()

print(f"Starting extraction across {total_urls} pages...")

for idx, url in enumerate(urls, start=1):
    delay = random.uniform(2, 4)
    print(f"[{idx}/{total_urls}] Downloading page... Sleeping {delay:.2f}s")
    time.sleep(delay)

    df_temp = scrape_finviz_curl(url)
    if df_temp is not None:
        df = pd.concat([df, df_temp], ignore_index=True)
    else:
        print(f"Skipped page due to download failure: {url}")

In [ ]:
if not df.empty and "Ticker" in df.columns:
    tickers_str = df["Ticker"].astype(str)

    # 1. Sanity Check: Ensure tickers have >= 2 characters before stripping
    short_tickers = tickers_str[tickers_str.str.len() < 2]
    if not short_tickers.empty:
        print(
            f"Warning: {len(short_tickers)} ticker(s) had fewer than 2 characters before modification:"
        )
        print(short_tickers.tolist())
    else:
        print("Sanity Check Passed: All tickers contain at least 2 characters.")

    # Remove the prefix character from tickers
    df["Ticker"] = tickers_str.str[1:]

    # 2. Check presence of SPY and AAPL after stripping
    unique_tickers = set(df["Ticker"])
    for check_symbol in ["SPY", "AAPL"]:
        found = check_symbol in unique_tickers
        print(f"Verification Check - Symbol '{check_symbol}' found: {found}")

    # Numeric conversion for appropriate data columns
    for col in df.columns:
        if col == "Ticker":
            continue
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().any():
            df[col] = converted

In [ ]:
parquet_filename = f"df_finviz_{current_date_pst}_stocks_etfs.parquet"
csv_filename = f"ticker_{current_date_pst}_stocks_etfs.csv"
zip_filename = f"{current_date_pst}_combined_files.zip"

In [ ]:
pst = pytz.timezone("America/Los_Angeles")
current_date_pst = datetime.datetime.now(pst).strftime("%Y-%m-%d")

parquet_filename = f"df_finviz_{current_date_pst}_stocks_etfs.parquet"
csv_filename = f"ticker_{current_date_pst}_stocks_etfs.csv"
zip_filename = f"{current_date_pst}_combined_files.zip"

parquet_path = output_dir / parquet_filename
csv_path = output_dir / csv_filename
zip_path = output_dir / zip_filename

# Save parquet and ticker CSV
df.to_parquet(parquet_path, engine="pyarrow", compression="zstd")
if "Ticker" in df.columns:
    df[["Ticker"]].to_csv(csv_path, header=False, index=False)

# Archive into ZIP
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(parquet_path, arcname=parquet_filename)
    if csv_path.exists():
        zipf.write(csv_path, arcname=csv_filename)

print(f"Files successfully generated and compressed at:\n{zip_path.resolve()}")

# Clean up loose files
parquet_path.unlink(missing_ok=True)
csv_path.unlink(missing_ok=True)

# Trigger automatic download to local PC Downloads folder if running in Colab
if IS_COLAB:
    from google.colab import files

    print("Initiating browser download to PC Downloads folder...")
    files.download(str(zip_path))

In [ ]:
df